# Orthomosaic — Monitor

**Optional: open in a separate tab while `01a_sfm_orthomosaic` is running to watch incremental mapping progress.**

Polls the `sparse/` model directory every 30 seconds and reports the number of registered
images and the latest snapshot count.

> **Runtime.** Runs on **Serverless environment 5**.

---

**Last Update:** September 21, 2026

## Setup

In [ ]:
%run ./config_nb

In [ ]:
import time as _time_monitor
from pathlib import Path

sparse_dir = Path(output_dir) / "sparse"
snap_dir   = Path(output_dir) / "snapshots"

# Adjust total_images to the QC-filtered count from 01a_sfm_orthomosaic.
_total = total_images
_poll_interval = 30  # seconds

print(f"Watching {sparse_dir}  (Ctrl-C or interrupt to stop)")
print(f"Expected images: {_total}")

try:
    while True:
        # Count registered images across all sparse model subdirs
        _registered = 0
        if sparse_dir.exists():
            for _model_dir in [d for d in sparse_dir.iterdir() if is_model_dir(d)]:
                _registered = max(_registered, count_registered(_model_dir))

        # Count snapshot dirs
        _snaps = len([d for d in snap_dir.iterdir() if d.is_dir()]) if snap_dir.exists() else 0

        _pct = 100 * _registered / _total if _total > 0 else 0
        print(
            f"[{_time_monitor.strftime('%H:%M:%S')}] "
            f"Registered: {_registered}/{_total} ({_pct:.0f}%)  |  "
            f"Snapshots: {_snaps}",
            flush=True,
        )
        if _registered >= _total:
            print("All images registered — mapping complete.")
            break
        _time_monitor.sleep(_poll_interval)
except KeyboardInterrupt:
    print("Monitor stopped.")